In this notebook, you will prepare the hyperparameters for starting a training loop for a sliding diffusion model or for a standard diffusion model.

First make the relevant imports, by calling this cell:

In [ ]:
import torch
from model import ContinuousMotionModel
from utils.debugger import Debugger
from model_training_loop import train
from diffusion_process_super import Diffusion
from diffusion_process_sliding import SlidingDiffusion
from diffusion_process_normal import NormalDiffusion
from torch.utils.data import DataLoader
from dataset.dataset import *
import utils.utils as utils

First, define the device (CPU or GPU/Cuda). It must be consistently synchronized across all training functions and model specifications, so it is the first thing to define.

In [ ]:
device = utils.get_device()

If you want to train a sliding diffusion model, you need to configure the hyperparameters specific to sliding diffusion, all of which relate to the diffusion object. These include the clean section length, the denoising section length, and the noising schedule. In the cell below, a few predefined hyperparameters are provided, but you are free to adjust them as needed.

In [ ]:
# For sliding diffusion:
num_clean_frames = 50
num_denoise_frames = 20

# as the seq_length is used in multiple places, we define it separately outside the train function call
seq_length = num_clean_frames + num_denoise_frames

if num_denoise_frames == 5:
    noise_schedule = Diffusion.linear_schedule(0.00005, 0.55)
if num_denoise_frames == 20:
    noise_schedule = Diffusion.linear_schedule(0.00015, 0.2)
if num_denoise_frames == 50:
    noise_schedule = Diffusion.linear_schedule(0.00020, 0.1)
if num_denoise_frames == 100:
    noise_schedule = Diffusion.linear_schedule(0.00020, 0.06)
if num_denoise_frames == 200:
    noise_schedule = Diffusion.linear_schedule(0.00025, 0.003)

diffusion_model = SlidingDiffusion(
    num_clean_frames = num_clean_frames,
    num_denoise_frames = num_denoise_frames,
    num_noise_frames = 0,
    num_timestep_stackings = 1, # You can experiment with number of stacking levels here
    noise_schedule = noise_schedule,
    device = device
)

If you want to train a standard diffusion model instead, you can run this cell to override the diffusion object created in the previous cell and set the hyperparameters for the standard diffusion process. These include parameters such as the number of time steps and the noising schedule.

In [ ]:
# as the seq_length is used in multiple places, we define it separately outside the train function call
seq_length = 70

diffusion_model = NormalDiffusion(
    num_timesteps = 100,
    sequence_length = seq_length,
    noise_schedule = Diffusion.linear_schedule(0.00015, 0.075),
    device = device
)

Once the diffusion type is defined, you can configure the training loop along with the objects it requires, such as the model and its specifications. We have designed the training function call to collect nearly all hyperparameters in one place, making it easier to review and modify them.

In [ ]:
seed_frames_length = 8  # Becouse the number of seed frames is required in multiple places, we define it separately outside the train function call

train(
    num_fgd_samples = 2048,
    experiment_collection_name = "experiment1",
    upload_model_check_point = False, # should upload the model checkpoint to wandb
    model_checkpoint_dir = "models/experiment1", # directory to save model checkpoints
    model = ContinuousMotionModel(
        diffusion = diffusion_model,
        pose_encoder = None, # AdvancedPoseEncoder.load_from_checkpoint("advanced_pose_encoder_ik_pca_64", device),
        number_of_styles = 17,
        gesture_length = seq_length,
        seed_length = seed_frames_length,
        audio_features_per_frame = 37,
        pose_features_per_frame = 345,
        condition_mask_probabilty = 0.1,
        number_of_attention_heads = 8,
        predict_full_duration = True,
        reinject_seed_style_full_t = False,
        debugger = Debugger(
            on = False,
            keys_for_printing_while_running = ["ALL"]
        ),
        device = device
    ),
    device = device,
    training_loader = DataLoader(
            GPUDataset(
                consolidated_file = "dataset/genea2023_dataset/trn/main-agent/consolidated.npz",
                seq_length = seq_length,
                seed_length = seed_frames_length,
                batch_size = 256,
                epoch_length = 1000,
                loading_encoded_data = False,
                include_vel_acc_features = False,
                device = device
            ),
            batch_size = 1,
            num_workers = 0,
            pin_memory = False
        ),
    val_loader = DataLoader(
            GPUDataset(
                consolidated_file = "dataset/genea2023_dataset/val/main-agent/consolidated.npz",
                seq_length = seq_length,
                seed_length = seed_frames_length,
                batch_size = 64,
                epoch_length = 30,
                loading_encoded_data = False,
                include_vel_acc_features = False,
                device = device
            ),
            batch_size = 1,
            num_workers = 0,
            pin_memory = False
        ),
    num_epochs = 2000,
    learning_rate = 0.00005,
    reconstruction_loss_weight = 4.0,
    variance_loss_weight = 0.1,
    velocity_loss_weight = 1.0,
    acceleration_loss_weight = 1.5,
    jerk_loss_weight= 0.2,
    latent_space_loss_weight = 0.0,
    category_weighting = {
        'fingers': 0.1,
        'arms': 2.0,
        'legs': 1.0,
        'spine': 2.0,
        'head': 1.0,
        'root': 2.0
    },
    visualize_step = 1
)